# Process Flowsheet: Mixer → Heater → CSTR

A simple process combining multiple unit operations into a flowsheet:

1. **Mixer** — blends a fresh feed stream with a recycle-like second stream
2. **Heater** — brings the mixed stream to reactor inlet temperature
3. **CSTR** — exothermic first-order reaction A → products with cooling jacket

This demonstrates how PathSim-Chem blocks compose into multi-unit simulations,
inspired by the reactor front-end of [MiniSim's Cumene Process](https://github.com/Nukleon84/MiniSim/blob/master/doc/Cumene%20Process.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('../pathsim_docs.mplstyle')

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_chem.process import Mixer, Heater, CSTR

## System Setup

**Fresh feed:** 0.05 m³/s at 300 K with 2.0 mol/m³ reactant A  
**Second stream:** 0.05 m³/s at 310 K (simulating a warm recycle)  
**Heater:** 200 kW duty to preheat the mixed stream  
**CSTR:** 1 m³ reactor, first-order Arrhenius kinetics, cooled by a 290 K jacket

In [ ]:
# Unit operations
mixer = Mixer()
heater = Heater(rho=1000.0, Cp=4184.0)  # water-like fluid
cstr = CSTR(
    V=1.0,          # reactor volume [m³]
    F=0.1,          # total flow [m³/s]
    k0=1e6,         # pre-exponential factor [1/s]
    Ea=50000.0,     # activation energy [J/mol]
    n=1.0,          # first-order reaction
    dH_rxn=-50000.0,  # exothermic [J/mol]
    rho=1000.0,
    Cp=4184.0,
    UA=500.0,       # heat transfer [W/K]
    C_A0=0.0,       # start empty
    T0=300.0,       # start at 300 K
)

# Feed sources
src_f1 = Source(lambda t: 0.05)    # fresh feed flow [m³/s]
src_t1 = Source(lambda t: 300.0)   # fresh feed temp [K]
src_f2 = Source(lambda t: 0.05)    # second stream flow [m³/s]
src_t2 = Source(lambda t: 310.0)   # second stream temp [K]

# Heater duty [W]
src_q = Source(lambda t: 200000.0)

# Reactant concentration in combined feed [mol/m³]
src_cin = Source(lambda t: 2.0)

# Coolant temperature [K]
src_tc = Source(lambda t: 290.0)

# Scopes for recording
sco_mix = Scope(labels=["F_mix", "T_mix"])
sco_heat = Scope(labels=["F_heat", "T_heated"])
sco_cstr = Scope(labels=["C_out", "T_reactor"])

## Wiring the Flowsheet

```
Feed 1 (F₁, T₁) ─┐
                   ├─ Mixer ─── Heater ─── CSTR ─── Products
Feed 2 (F₂, T₂) ─┘     ↑         ↑          ↑
                         │         Q          T_coolant
```

Note: The CSTR's internal flow rate `F` is set at construction. Here the mixer's output flow
is used for monitoring, while the CSTR uses its own `F=0.1` parameter for residence time.

In [ ]:
sim = Simulation(
    [
        src_f1, src_t1, src_f2, src_t2, src_q, src_cin, src_tc,
        mixer, heater, cstr,
        sco_mix, sco_heat, sco_cstr,
    ],
    [
        # Mixer inputs: (F_1, T_1, F_2, T_2)
        Connection(src_f1, mixer),
        Connection(src_t1, mixer[1]),
        Connection(src_f2, mixer[2]),
        Connection(src_t2, mixer[3]),

        # Mixer outputs -> Heater inputs: (F, T_in)
        Connection(mixer, heater),          # F_out -> F
        Connection(mixer[1], heater[1]),    # T_out -> T_in

        # Heater heat duty
        Connection(src_q, heater[2]),       # Q

        # Heater output temperature -> CSTR inlet temperature
        Connection(src_cin, cstr),          # C_in
        Connection(heater[1], cstr[1]),     # T_heated -> T_in

        # CSTR coolant
        Connection(src_tc, cstr[2]),        # T_c

        # Recording
        Connection(mixer, sco_mix),
        Connection(mixer[1], sco_mix[1]),
        Connection(heater, sco_heat),
        Connection(heater[1], sco_heat[1]),
        Connection(cstr, sco_cstr),
        Connection(cstr[1], sco_cstr[1]),
    ],
    dt=0.1,
    log=True,
)

sim.run(100)

## Results: Temperature Profile Through the Plant

Track the temperature at each stage: after mixing, after heating, and inside the reactor.

In [ ]:
t_mix, [F_mix, T_mix] = sco_mix.read()
t_heat, [F_heat, T_heated] = sco_heat.read()
t_cstr, [C_out, T_reactor] = sco_cstr.read()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Temperature evolution
ax1.plot(t_mix, T_mix, label="After Mixer")
ax1.plot(t_heat, T_heated, label="After Heater")
ax1.plot(t_cstr, T_reactor, label="Reactor (CSTR)")
ax1.axhline(290, color="blue", ls="--", alpha=0.3, label="Coolant")
ax1.set_xlabel("Time [s]")
ax1.set_ylabel("Temperature [K]")
ax1.set_title("Temperature Profile")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Reactor concentration
ax2.plot(t_cstr, C_out, color="tab:red")
ax2.set_xlabel("Time [s]")
ax2.set_ylabel("Concentration $C_A$ [mol/m³]")
ax2.set_title("Reactor Outlet Concentration")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Steady-State Summary

In [ ]:
print("Process Steady State")
print("=" * 45)
print(f"{'Unit':15s} {'Flow [m³/s]':>12s} {'T [K]':>10s}")
print("-" * 45)
print(f"{'Fresh feed':15s} {'0.05':>12s} {'300.0':>10s}")
print(f"{'Second stream':15s} {'0.05':>12s} {'310.0':>10s}")
print(f"{'Mixer outlet':15s} {F_mix[-1]:12.4f} {T_mix[-1]:10.2f}")
print(f"{'Heater outlet':15s} {F_heat[-1]:12.4f} {T_heated[-1]:10.2f}")
print(f"{'CSTR outlet':15s} {'—':>12s} {T_reactor[-1]:10.2f}")
print(f"\nReactor C_A = {C_out[-1]:.4f} mol/m³ (feed = 2.0 mol/m³)")
print(f"Conversion = {(1 - C_out[-1]/2.0)*100:.1f}%")

The mixer blends both feeds to an intermediate temperature (~305 K). The heater raises it
further. The CSTR then reaches a steady state where the exothermic reaction heat is balanced
by cooling jacket removal, and the conversion depends on the residence time and Arrhenius rate.